# Figure 3: Tradeoff axes defined by PLS regression coefficients

In [1]:
import numpy as np
import pandas as pd

import bokeh.io
import bokeh.plotting
from bokeh.models import ColumnDataSource
bokeh.io.output_notebook()

import iqplot

import sys
sys.path.insert(0, '../analysis')
from pls_ecoli import run_pls

Loading BokehJS ...

## Figure 3A: ribosomal vs metabolic proteome fraction (Belliveau et al., 2021)

Adapted from Github available from Belliveau et al. No extra processing steps were done.

## Figure 3B: PLS-derived tradeoff axes for E. coli mRNA data

### Plot tradeoff axes

**Load growth data**

In [2]:
df_hwa_gr = pd.read_excel('../data/e.coli/science.abk2066_table_s3.xlsx', sheet_name = 0)
growth_map = df_hwa_gr.set_index("Sample ID")["Growth rate (1/h)"]

**Load mRNA data**

In [3]:
df_hwa_mrna = pd.read_excel("../data/e.coli/science.abk2066_table_s3.xlsx", sheet_name = 1)
df_hwa_mrna.head(2)

,gene,locus,gene length (nt),c4,c4_1,c3_1,c3,c2,c2_1,c1,...,r3_1,r5,r4,r2_1,r3,r2,r1_1,r1,r0,r0_1
0,aaeA,b3241,933,0.000003,0.000002,0.000002,0.000003,0.000003,0.000003,0.000003,...,0.000005,0.000005,0.000005,0.000005,0.000004,0.000005,0.000003,0.000004,0.000003,0.000003
1,aaeB,b3240,1968,0.000003,0.000002,0.000003,0.000002,0.000002,0.000002,0.000002,...,0.000006,0.000005,0.000005,0.000005,0.000005,0.000004,0.000004,0.000004,0.000003,0.000003


**Process mRNA df and add growth values**

In [4]:
df_hwa_mrna = df_hwa_mrna.drop(['locus', 'gene length (nt)'], axis = 1)
df_hwa_mrna_T = df_hwa_mrna.T
df_hwa_mrna_T.columns = df_hwa_mrna_T.iloc[0].values

# drop sample with no growth rate
df_hwa_mrna_T = df_hwa_mrna_T.drop(['gene', 'a4_1.1']) 

# add growth rate values
df_hwa_mrna_T['growth_rate'] = df_hwa_mrna_T.index.map(growth_map)

df_hwa_mrna_T.head(2)

,aaeA,aaeB,aaeR,aaeX,aas,aat,abgA,abgB,abgR,abgT,...,znuA,znuB,znuC,zraP,zraR,zraS,zupT,zur,zwf,growth_rate
c4,0.000003,0.000003,0.000021,0.000016,0.000029,0.000049,0.000002,0.000005,0.00005,0.000004,...,0.000084,0.000026,0.000042,0.000027,0.000007,0.00001,0.00008,0.000039,0.000166,0.30
c4_1,0.000002,0.000002,0.000017,0.000008,0.000032,0.000047,0.000003,0.000004,0.000043,0.000005,...,0.000085,0.00003,0.000038,0.000005,0.000007,0.000008,0.000092,0.00004,0.000182,0.39


**Define separate dfs for each condition**

In [5]:
# carbon
df_c = df_hwa_mrna_T[df_hwa_mrna_T.index.str.startswith('c')]
# ammonia
df_a = df_hwa_mrna_T[df_hwa_mrna_T.index.str.startswith('a')]
# chloramphenicol
df_r = df_hwa_mrna_T[df_hwa_mrna_T.index.str.startswith('r')]

**Plot**

In [6]:
beta_df, p_scatter, p_hist = run_pls(df_c, t=0.00006, condition = 'Carbon')

G+/G- correlation: -0.8477277828305335 | G+ genes: 961 | G- genes: 1646


In [7]:
beta_df, p_scatter, p_hist = run_pls(df_a, t=0.00006, condition = 'Ammonia')

G+/G- correlation: -0.7887572905280551 | G+ genes: 508 | G- genes: 1652


In [8]:
beta_df, p_scatter, p_hist = run_pls(df_r, t=0.00006, condition = 'Chloramphenicol')

G+/G- correlation: -0.9247277959874242 | G+ genes: 1474 | G- genes: 1131


### Plot GSEA results

First, I used the Broad GSEA 